# Hamiltonian Mechanics of Albatross Migration

**Derivation of the Euler–Lagrange equations, the Pontryagin Hamiltonian, and the time-invariant canonical system for DS-driven macro-scale migration. All notation follows standard dynamic optimisation theory.**

---

### Outline

1. Problem Setup — State, Control, and Dynamics
2. The Objective Functional and the Lagrangian
3. The Euler–Lagrange Equations via the Variational Principle
4. The Pontryagin Hamiltonian and the Maximality Condition
5. The Legendre Transform — connecting Lagrangian and Hamiltonian views
6. The Canonical Equations — States and Co-states
7. Time-Invariance and Conservation of the Hamiltonian
8. Connection to `migration_hamiltonian.py`

## 1. Problem Setup — State, Control, and Dynamics

### Energy-conserving DS cycles justify a macro-scale description

A complete dynamic soaring (DS) cycle is energy-neutral: the bird returns to the same airspeed and altitude, having extracted from the wind shear exactly the energy lost to drag. Formally:

$$\Delta E_{\mathrm{cycle}} = \oint \mathbf{F}_{\mathrm{aero}} \cdot d\mathbf{r} = 0$$

Because each cycle is energy-conserving, the fast within-cycle dynamics (timescale $\sim 20\ \mathrm{s}$) can be averaged out. The macro-scale migration (timescale $\sim$ days) is then fully described by a single quantity per cycle: the **net ground displacement vector** $\bar{\mathbf{v}}$, i.e. the cycle-averaged ground velocity.

The set of all achievable $\bar{\mathbf{v}}$ at a given location and wind speed defines the **velocity hull** $\mathcal{V}(\mathbf{q}, t)$ — a convex, compact subset of $\mathbb{R}^2$ computed from DS flight mechanics. The hull depends on time through the local wind $\mathbf{w}(\mathbf{q}, t)$, which in the numerical implementation is the full ERA5 hourly wind field.

### State and control

We define:

$$\mathbf{q}(t) = \begin{pmatrix} q_1(t) \\ q_2(t) \end{pmatrix} \in \mathbb{R}^2 \quad \text{(Eastward, Northward position)}$$

$$\mathbf{v}(t) \in \mathcal{V}(\mathbf{q}(t), t) \quad \text{(cycle-averaged ground velocity — the control)}$$

### Dynamics

The state equation is:

$$\dot{\mathbf{q}}(t) = \mathbf{v}(t), \qquad \mathbf{v}(t) \in \mathcal{V}(\mathbf{q}(t),\, t), \qquad \mathbf{q}(0) = \mathbf{q}_0$$

This is a **nonlinear, time-varying control system** with a set-valued control constraint. The hull $\mathcal{V}(\mathbf{q}, t)$ captures all the aerodynamics and wind effects implicitly — the macro-scale model needs no further physics.

## 2. The Objective Functional and the Lagrangian

### Objective

Fix a migration direction $\mathbf{d} \in S^1$ (unit vector). We want to **maximise the distance covered in direction $\mathbf{d}$** over a fixed time horizon $T$:

$$\max_{\mathbf{v}(\cdot)} \quad J[\mathbf{v}] = \mathbf{d} \cdot \mathbf{q}(T)$$

Using $\mathbf{d} \cdot \mathbf{q}(T) = \mathbf{d} \cdot \mathbf{q}_0 + \int_0^T \mathbf{d} \cdot \dot{\mathbf{q}}\, dt$, the objective becomes:

$$\max_{\mathbf{v}(\cdot)} \int_0^T \mathbf{d} \cdot \mathbf{v}(t)\, dt$$

### The Lagrangian

In dynamic optimisation the **Lagrangian** (or running cost) is the integrand of the objective. Here:

$$L(\mathbf{q}, \mathbf{v}) = \mathbf{d} \cdot \mathbf{v}$$

The total objective is $J = \int_0^T L(\mathbf{q}, \mathbf{v})\, dt$, and we seek the trajectory $\mathbf{q}(\cdot)$ and control $\mathbf{v}(\cdot)$ that maximise it subject to the dynamics $\dot{\mathbf{q}} = \mathbf{v}$.

### Incorporating the dynamics as a constraint — the augmented Lagrangian

Introduce a **Lagrange multiplier** $\mathbf{p}(t) \in \mathbb{R}^2$ to enforce the state equation $\dot{\mathbf{q}} = \mathbf{v}$ at every instant. The augmented (Pontryagin) Lagrangian is:

$$\mathcal{L}(\mathbf{q}, \dot{\mathbf{q}}, \mathbf{v}, \mathbf{p}) = L(\mathbf{q}, \mathbf{v}) + \mathbf{p}^\top(\dot{\mathbf{q}} - \mathbf{v})$$

$$= \mathbf{d} \cdot \mathbf{v} + \mathbf{p} \cdot \dot{\mathbf{q}} - \mathbf{p} \cdot \mathbf{v}$$

$$= \mathbf{p} \cdot \dot{\mathbf{q}} + (\mathbf{d} - \mathbf{p}) \cdot \mathbf{v}$$

The augmented action is:

$$S = \int_0^T \mathcal{L}(\mathbf{q}, \dot{\mathbf{q}}, \mathbf{v}, \mathbf{p})\, dt$$

Finding the stationary point of $S$ with respect to $(\mathbf{q}, \mathbf{v}, \mathbf{p})$ simultaneously enforces the dynamics and the optimality conditions.

## 3. The Euler–Lagrange Equations via the Variational Principle

We now require the augmented action $S = \int_0^T \mathcal{L}\, dt$ to be stationary under independent perturbations of $\mathbf{q}$, $\mathbf{v}$, and $\mathbf{p}$.

---

### Variation with respect to $\mathbf{p}$ (enforces the dynamics)

Let $\mathbf{p} \to \mathbf{p} + \epsilon\, \boldsymbol{\mu}$. Then:

$$\delta_\mathbf{p} S = \int_0^T \boldsymbol{\mu} \cdot (\dot{\mathbf{q}} - \mathbf{v})\, dt = 0 \quad \forall\, \boldsymbol{\mu}$$

$$\boxed{\dot{\mathbf{q}} = \mathbf{v}}$$

This simply recovers the state equation — the multiplier $\mathbf{p}$ acts as the constraint that forces $\mathbf{v}$ to equal $\dot{\mathbf{q}}$.

---

### Variation with respect to $\mathbf{q}$ (gives the co-state equation)

Let $\mathbf{q} \to \mathbf{q} + \epsilon\, \boldsymbol{\eta}$ with $\boldsymbol{\eta}(0) = 0$ (fixed initial condition) and $\boldsymbol{\eta}(T)$ free. Expanding $\mathcal{L}$:

$$\delta_\mathbf{q} S = \int_0^T \left[ \frac{\partial \mathcal{L}}{\partial \mathbf{q}} \cdot \boldsymbol{\eta} + \frac{\partial \mathcal{L}}{\partial \dot{\mathbf{q}}} \cdot \dot{\boldsymbol{\eta}} \right] dt = 0$$

The second term is integrated by parts:

$$\int_0^T \frac{\partial \mathcal{L}}{\partial \dot{\mathbf{q}}} \cdot \dot{\boldsymbol{\eta}}\, dt = \left[ \frac{\partial \mathcal{L}}{\partial \dot{\mathbf{q}}} \cdot \boldsymbol{\eta} \right]_0^T - \int_0^T \frac{d}{dt}\!\left(\frac{\partial \mathcal{L}}{\partial \dot{\mathbf{q}}}\right) \cdot \boldsymbol{\eta}\, dt$$

Since $\mathcal{L} = \mathbf{p} \cdot \dot{\mathbf{q}} + (\mathbf{d} - \mathbf{p}) \cdot \mathbf{v}$, we have:
- $\partial \mathcal{L}/\partial \dot{\mathbf{q}} = \mathbf{p}$
- $\partial \mathcal{L}/\partial \mathbf{q} = \mathbf{0}$ (since $L = \mathbf{d} \cdot \mathbf{v}$ and the hull $\mathcal{V}(\mathbf{q})$ is implicitly encoded via the control constraint, handled separately via the Pontryagin condition)

The stationarity condition becomes:

$$\int_0^T \left(-\dot{\mathbf{p}}\right) \cdot \boldsymbol{\eta}\, dt + \left[\mathbf{p} \cdot \boldsymbol{\eta}\right]_0^T = 0$$

The boundary term at $t = T$: since $\boldsymbol{\eta}(T)$ is free and the terminal cost is $\mathbf{d} \cdot \mathbf{q}(T)$, the transversality condition gives $\mathbf{p}(T) = \mathbf{d}$.

For the integral to vanish for all $\boldsymbol{\eta}$, we need:

$$\boxed{\dot{\mathbf{p}} = -\frac{\partial H^*}{\partial \mathbf{q}}}$$

where we have anticipated that $\mathbf{p}$ will be identified as the co-state driving the Hamiltonian gradient — this is derived explicitly in Section 4. When the velocity hull depends on position (through the local wind), $\partial H^*/\partial \mathbf{q} \neq 0$ and the co-state evolves.

---

### Variation with respect to $\mathbf{v}$ (gives the optimality condition)

Let $\mathbf{v} \to \mathbf{v} + \epsilon\, \boldsymbol{\xi}$ (with $\mathbf{v} + \epsilon\boldsymbol{\xi} \in \mathcal{V}(\mathbf{q})$):

$$\delta_\mathbf{v} S = \int_0^T (\mathbf{d} - \mathbf{p}) \cdot \boldsymbol{\xi}\, dt = 0$$

For this to hold at the **interior** of $\mathcal{V}$ for all perturbations $\boldsymbol{\xi}$, we would need $\mathbf{p} = \mathbf{d}$. But since $\mathbf{v}$ is constrained to $\mathcal{V}(\mathbf{q})$, interior stationarity does not hold — instead, the optimum lies on the **boundary** $\partial\mathcal{V}$, and we must use the constrained maximisation condition derived in Section 4.

## 4. The Pontryagin Hamiltonian and the Maximality Condition

### Defining the Hamiltonian

The **Pontryagin Hamiltonian** is defined by combining the running cost $L$ with the dynamics $f(\mathbf{q}, \mathbf{v}) = \mathbf{v}$ weighted by the co-state $\mathbf{p}$:

$$H(\mathbf{q}, \mathbf{p}, \mathbf{v}) = \mathbf{p} \cdot f(\mathbf{q}, \mathbf{v}) + L(\mathbf{q}, \mathbf{v}) = \mathbf{p} \cdot \mathbf{v} + \mathbf{d} \cdot \mathbf{v} = (\mathbf{p} + \mathbf{d}) \cdot \mathbf{v}$$

> **Note on sign convention:** here we use the economist/maximisation convention where $H = \mathbf{p} \cdot f + L$. The physicist/minimisation convention has $H = \mathbf{p} \cdot f - L$; both are equivalent up to sign of $\mathbf{p}$. We will absorb $\mathbf{d}$ into $\mathbf{p}$ by redefining the co-state below.

### Redefining the co-state

Since $\mathbf{d}$ is a fixed constant, we redefine $\tilde{\mathbf{p}} = \mathbf{p} + \mathbf{d}$ so that:

$$H(\mathbf{q}, \tilde{\mathbf{p}}, \mathbf{v}) = \tilde{\mathbf{p}} \cdot \mathbf{v}$$

Dropping the tilde, the Hamiltonian is simply:

$$\boxed{H(\mathbf{q}, \mathbf{p}, \mathbf{v}) = \mathbf{p} \cdot \mathbf{v}}$$

with the transversality condition $\mathbf{p}(T) = \mathbf{d}$ carrying the information about the terminal objective.

### The Maximality Condition (Pontryagin Maximum Principle)

The **Pontryagin Maximum Principle** states that along any optimal trajectory, the control $\mathbf{v}^*(t)$ must **pointwise maximise** the Hamiltonian over all admissible controls:

$$\mathbf{v}^*(t) = \underset{\mathbf{v}\,\in\,\mathcal{V}(\mathbf{q}(t))}{\arg\max}\; H(\mathbf{q}(t), \mathbf{p}(t), \mathbf{v}) = \underset{\mathbf{v}\,\in\,\mathcal{V}(\mathbf{q}(t))}{\arg\max}\; \mathbf{p}(t) \cdot \mathbf{v}$$

The **maximised Hamiltonian** is then:

$$H^*(\mathbf{q}, \mathbf{p}) = \max_{\mathbf{v}\,\in\,\mathcal{V}(\mathbf{q})}\; \mathbf{p} \cdot \mathbf{v}$$

This is the **support function** of the convex set $\mathcal{V}(\mathbf{q})$ evaluated at $\mathbf{p}$. The maximiser $\mathbf{v}^*$ is the point on the boundary $\partial\mathcal{V}(\mathbf{q})$ whose outward normal is aligned with $\mathbf{p}$.

### Co-state equation (explicit)

The co-state evolves according to:

$$\dot{\mathbf{p}} = -\frac{\partial H^*}{\partial \mathbf{q}} = -\frac{\partial}{\partial \mathbf{q}} \max_{\mathbf{v}\in\mathcal{V}(\mathbf{q})} \mathbf{p} \cdot \mathbf{v}$$

The hull $\mathcal{V}(\mathbf{q})$ depends on position through the local wind $\mathbf{w}(\mathbf{q})$. For the simplified case of a circular hull (bird flies at airspeed $v_a$ in wind $\mathbf{w}(\mathbf{q})$), $H^* = v_a |\mathbf{p}| + \mathbf{p} \cdot \mathbf{w}(\mathbf{q})$, giving:

$$\dot{p}_i = -\sum_j p_j \frac{\partial w_j}{\partial q_i} = -\left[(\nabla \mathbf{w})^\top \mathbf{p}\right]_i$$

Written out:

$$\dot{p}_1 = -p_1 \frac{\partial w_1}{\partial q_1} - p_2 \frac{\partial w_2}{\partial q_1}, \qquad \dot{p}_2 = -p_1 \frac{\partial w_1}{\partial q_2} - p_2 \frac{\partial w_2}{\partial q_2}$$

**The co-state is driven by the Jacobian of the wind field.** In a spatially uniform wind ($\nabla \mathbf{w} = 0$), $\dot{\mathbf{p}} = 0$ and the co-state remains constant throughout the migration.

## 5. The Legendre Transform — Connecting Lagrangian and Hamiltonian

The Lagrangian and Hamiltonian formulations are related by the **Legendre transform**. This section shows the connection explicitly.

### From Lagrangian to Hamiltonian

Recall the augmented Lagrangian from Section 2 (after substituting $\dot{\mathbf{q}} = \mathbf{v}$ from the state equation):

$$\mathcal{L}(\mathbf{q}, \dot{\mathbf{q}}, \mathbf{p}) = \mathbf{p} \cdot \dot{\mathbf{q}} - H^*(\mathbf{q}, \mathbf{p})$$

This is the **Hamilton–Pontryagin Lagrangian**. It is already in the form $\mathcal{L} = \mathbf{p} \cdot \dot{\mathbf{q}} - H^*$, which is the standard relationship between the two descriptions.

The Legendre transform is defined as:

$$H^*(\mathbf{q}, \mathbf{p}) = \sup_{\dot{\mathbf{q}}} \left[ \mathbf{p} \cdot \dot{\mathbf{q}} - \mathcal{L}(\mathbf{q}, \dot{\mathbf{q}}) \right]$$

The stationarity condition $\partial/\partial \dot{\mathbf{q}}\left[\mathbf{p} \cdot \dot{\mathbf{q}} - \mathcal{L}\right] = 0$ gives:

$$\mathbf{p} = \frac{\partial \mathcal{L}}{\partial \dot{\mathbf{q}}}$$

which defines $\mathbf{p}$ as the **generalised momentum** conjugate to $\mathbf{q}$ — exactly what the Euler–Lagrange equations identified as the co-state.

### Inverse transform: Hamiltonian → Lagrangian

Going the other way, the Lagrangian is recovered from the Hamiltonian by:

$$\mathcal{L}(\mathbf{q}, \dot{\mathbf{q}}) = \inf_{\mathbf{p}} \left[ \mathbf{p} \cdot \dot{\mathbf{q}} - H^*(\mathbf{q}, \mathbf{p}) \right]$$

The stationarity condition $\partial/\partial \mathbf{p}\left[\mathbf{p} \cdot \dot{\mathbf{q}} - H^*\right] = 0$ gives:

$$\dot{\mathbf{q}} = \frac{\partial H^*}{\partial \mathbf{p}}$$

which is the **state equation** — one half of the canonical equations.

### Summary of the duality

The two formulations encode the same optimal trajectory through different variables:

| | Lagrangian | Hamiltonian |
|---|---|---|
| Primary variable | $\dot{\mathbf{q}}$ (velocity) | $\mathbf{p}$ (co-state / momentum) |
| Equation of motion | $\frac{d}{dt}\frac{\partial \mathcal{L}}{\partial \dot{\mathbf{q}}} = \frac{\partial \mathcal{L}}{\partial \mathbf{q}}$ (2nd order in $\mathbf{q}$) | $\dot{\mathbf{q}} = \nabla_\mathbf{p} H^*$, $\dot{\mathbf{p}} = -\nabla_\mathbf{q} H^*$ (1st order system) |
| Phase space | Tangent bundle $T\mathcal{Q}$: $(\mathbf{q}, \dot{\mathbf{q}})$ | Cotangent bundle $T^*\mathcal{Q}$: $(\mathbf{q}, \mathbf{p})$ |
| Conservation | Not obvious | $H^* = \text{const}$ immediately (Section 7) |

The Hamiltonian formulation is preferred for optimal control because conservation and symplectic structure are explicit.

## 6. The Canonical Equations — States and Co-states

Collecting all results, the optimal migration system is a **4-dimensional first-order ODE** on the phase space $(\mathbf{q}, \mathbf{p}) \in \mathbb{R}^2 \times \mathbb{R}^2$.

### Variables

| Variable | Dimension | Role | Physical meaning |
|----------|-----------|------|-----------------|
| $\mathbf{q}(t) = (q_1, q_2)^\top$ | **State** | 2 | Geographic position (East, North) |
| $\mathbf{p}(t) = (p_1, p_2)^\top$ | **Co-state** | 2 | Shadow price of position; gradient of value function |

### The canonical equations

$$\boxed{\dot{\mathbf{q}} = \phantom{-}\frac{\partial H^*}{\partial \mathbf{p}} = \mathbf{v}^*(\mathbf{q}, \mathbf{p})} \qquad \text{(state equation)}$$

$$\boxed{\dot{\mathbf{p}} = -\frac{\partial H^*}{\partial \mathbf{q}}} \qquad \text{(co-state equation)}$$

### Boundary conditions

- $\mathbf{q}(0) = \mathbf{q}_0$ — known start position (e.g. Crozet Island)
- $\mathbf{p}(T) = \mathbf{d}$ — **transversality condition**: at the end of the journey, the co-state equals the migration direction

This makes the full problem a **two-point boundary value problem (TPBVP)** in $(\mathbf{q}, \mathbf{p})$.

### Physical interpretation of the co-state

$\mathbf{p}(t)$ is the gradient of the **value function** $V(\mathbf{q}, t)$ with respect to position:

$$\mathbf{p}(t) = \frac{\partial V}{\partial \mathbf{q}}\bigg|_{\mathbf{q}(t),\, t}$$

where $V(\mathbf{q}, t) = \max_\mathbf{v} \int_t^T \mathbf{d} \cdot \mathbf{v}\, d\tau$ is the maximum remaining migration distance achievable from $\mathbf{q}$ at time $t$.

Intuitively, $p_i$ is the marginal value of a unit displacement in direction $i$ — how much extra migration distance the bird gains if it were nudged by $\epsilon$ in direction $i$ right now.

### The HJB equation

The value function $V(\mathbf{q}, t)$ satisfies the **Hamilton–Jacobi–Bellman (HJB) equation**:

$$-\frac{\partial V}{\partial t} = H^*\!\left(\mathbf{q},\, \nabla_\mathbf{q} V\right) = \max_{\mathbf{v}\in\mathcal{V}(\mathbf{q})} \nabla_\mathbf{q} V \cdot \mathbf{v}$$

with terminal condition $V(\mathbf{q}, T) = \mathbf{d} \cdot \mathbf{q}$. The optimal co-state trajectory $\mathbf{p}(t) = \nabla_\mathbf{q} V(\mathbf{q}(t), t)$ is a characteristic of this PDE — which is precisely why the canonical ODE system is called the **method of characteristics** for the HJB equation.

## 7. Time-Invariance and Conservation of the Hamiltonian

### The general (time-varying) case

When the wind field depends on time — as it does in the code, which uses hourly ERA5 data — the velocity hull $\mathcal{V}(\mathbf{q}, t)$ and the maximised Hamiltonian both carry explicit time dependence:

$$H^*(\mathbf{q}, \mathbf{p}, t) = \max_{\mathbf{v}\,\in\,\mathcal{V}(\mathbf{q},\,t)}\; \mathbf{p} \cdot \mathbf{v}$$

The total time derivative along a solution is:

$$\frac{d H^*}{dt} = \frac{\partial H^*}{\partial \mathbf{q}} \cdot \dot{\mathbf{q}} + \frac{\partial H^*}{\partial \mathbf{p}} \cdot \dot{\mathbf{p}} + \frac{\partial H^*}{\partial t}$$

Substituting the canonical equations as before, the first two terms cancel, leaving:

$$\frac{d H^*}{dt} = \frac{\partial H^*}{\partial t} = \frac{\partial}{\partial t}\max_{\mathbf{v}\,\in\,\mathcal{V}(\mathbf{q},\,t)}\; \mathbf{p} \cdot \mathbf{v}$$

This is generally **non-zero**: as the wind changes, so does the maximum achievable migration speed, and $H^*$ is not conserved. The system is **non-autonomous**.

---

### The time-invariant special case — and when it applies

If the wind field is replaced by a **time-averaged mean** $\bar{\mathbf{w}}(\mathbf{q})$, then $\partial H^*/\partial t = 0$ and the system becomes autonomous. In that case:

$$\frac{d H^*}{dt} = 0 \qquad \Longrightarrow \qquad \boxed{H^*(\mathbf{q}(t), \mathbf{p}(t)) = \text{const}}$$

This conservation holds rigorously only for the time-averaged model, **not** for the hourly ERA5 simulation. It is a useful theoretical result for understanding the structure of the problem and for constructing analytical approximations, but the numerical code operates in the time-varying regime.

---

### Proof of conservation (for the time-invariant case)

With $\partial H^*/\partial t = 0$, the total derivative reduces to:

$$\frac{d H^*}{dt} = \underbrace{\frac{\partial H^*}{\partial \mathbf{q}} \cdot \frac{\partial H^*}{\partial \mathbf{p}}}_{\text{term A}} - \underbrace{\frac{\partial H^*}{\partial \mathbf{p}} \cdot \frac{\partial H^*}{\partial \mathbf{q}}}_{\text{term B}} = 0$$

Terms A and B are the same scalar, so they cancel exactly. This is a direct consequence of the antisymmetry of the canonical equations, and holds for **any** autonomous Hamiltonian system.

---

### Summary

| Wind model | $\partial H^*/\partial t$ | $H^*$ conserved? |
|------------|--------------------------|------------------|
| Hourly ERA5 (as in the code) | $\neq 0$ | No — $H^*$ varies as wind changes |
| Seasonal mean (time-averaged) | $= 0$ | Yes — canonical conservation holds |

The micro-scale DS energy cycle is still conserved regardless: that is a property of the fast dynamics, not the macro-scale wind model.

| Level | Invariance | Conserved quantity |
|-------|------------|--------------------|
| Micro (DS cycle) | Cycle is periodic in fast time | Mechanical energy $E = \frac{1}{2}mv_a^2 + mgh$ |
| Macro (time-averaged wind) | Wind field time-invariant | Migration Hamiltonian $H^*(\mathbf{q},\mathbf{p})$ |
| Macro (hourly ERA5 wind) | Wind field time-varying | $H^*$ not conserved; full HJB must be solved |

## 8. Connection to `migration_hamiltonian.py`

### Wind model used during time marching

The code uses `ERA5WindInterpolator`, which loads the **1-hourly ERA5 files** (`era5_1h_so_*.nc`) and at every RK4 stage calls:

```python
u10, v10 = era5.query(lat, lon, t)   # t is the current Unix timestamp
```

The interpolator performs **bilinear spatial + linear temporal interpolation** between the two bracketing hourly snapshots. Since RK4 evaluates the right-hand side at $t$, $t + \Delta t/2$, and $t + \Delta t$, the wind seen by the integrator changes continuously throughout the march. The system is therefore genuinely **non-autonomous**: $\mathcal{V}(\mathbf{q}, t)$ and $H^*(\mathbf{q}, \mathbf{p}, t)$ vary with time, and $H^*$ is not conserved.

### What the code implements

For each of $N_\mathrm{dirs}$ fixed directions $\mathbf{d}_k$, the state ODE is integrated with the co-state held constant:

$$\dot{\mathbf{q}}_k(t) = \underset{\mathbf{v}\in\mathcal{V}(\mathbf{q}_k,\,t)}{\arg\max}\; \mathbf{d}_k \cdot \mathbf{v}, \qquad k = 1, \ldots, N_\mathrm{dirs}$$

The collection of endpoints $\{\mathbf{q}_k(T)\}$ traces out the **reachability envelope** at time $T$.

### Relationship to the full canonical system

The fixed-co-state approximation sets $\dot{\mathbf{p}} = 0$, which in the time-varying setting corresponds to neglecting both the spatial wind gradient and the temporal change in the hull:

$$\dot{\mathbf{p}} = -\frac{\partial H^*}{\partial \mathbf{q}} \approx 0$$

This is justified when spatial wind gradients are small over the trajectory length — a reasonable approximation for the large-scale Southern Ocean circulation.

### What the full system would require

The full non-autonomous optimal control problem requires solving a **time-varying TPBVP**: integrate both

$$\dot{\mathbf{q}} = \frac{\partial H^*}{\partial \mathbf{p}}(\mathbf{q}, \mathbf{p}, t), \qquad \dot{\mathbf{p}} = -\frac{\partial H^*}{\partial \mathbf{q}}(\mathbf{q}, \mathbf{p}, t)$$

forward from $\mathbf{q}(0) = \mathbf{q}_0$ and backward from $\mathbf{p}(T) = \mathbf{d}$ simultaneously — a shooting method or direct collocation over the hourly ERA5 field. The time-varying HJB equation would also need to carry the $\partial V/\partial t$ term:

$$-\frac{\partial V}{\partial t} = H^*\!\left(\mathbf{q},\, \nabla_\mathbf{q} V,\, t\right)$$

| | **Full non-autonomous system** | **Code implementation** |
|---|---|---|
| Wind model | ERA5 hourly, $\mathbf{w}(\mathbf{q},t)$ | ✓ ERA5 hourly, bilinear + linear interp. |
| State $\mathbf{q}$ | evolves via $\dot{\mathbf{q}} = \nabla_\mathbf{p} H^*$ | ✓ integrated (RK4) |
| Co-state $\mathbf{p}$ | evolves via $\dot{\mathbf{p}} = -\nabla_\mathbf{q} H^*$ | ✗ fixed at $\mathbf{d}_k$ |
| $H^*$ conservation | Not guaranteed ($\partial H^*/\partial t \neq 0$) | N/A — co-state fixed |
| TPBVP | Yes, shooting or collocation needed | ✗ replaced by greedy IVP sweep |